# PrevMaq — Previsão de Falhas em Máquinas Industriais

**Projeto Avaliativo — Módulo 1 · Desenvolvimento de IA para Análise Preditiva [T3] · SENAI/SC**

**Objetivo:** construir um pipeline preditivo que classifique se um equipamento vai **falhar** (`falha_maquina = 1`) ou **operar normalmente** (`falha_maquina = 0`), comparando um **KNN** e uma **Árvore de Decisão** pela acurácia no conjunto de teste.

**Base de dados:** `data/raw/manutencao_preditiva.csv` — dicionário em `docs/dicionario_de_dados.md`.

**Regra da Engenharia:** as colunas `falha_twf`, `falha_hdf`, `falha_pwf`, `falha_osf` e `falha_rnf` são só para consulta e **não podem** entrar em X.

| Fase | Conteúdo |
|---|---|
| 1 | Análise Exploratória (EDA) |
| 2 | Limpeza e Tratamento de Dados |
| 3 | Feature Engineering |
| 4 | Divisão e Balanceamento |
| 5 | Escalonamento (StandardScaler) |
| 6 | Ajuste de Parâmetros e Combate ao Overfitting |
| 7 | Avaliação da Acurácia e Veredito Final |

## 0. Configuração inicial

Importe as bibliotecas usadas no projeto e carregue a base com **caminho relativo** (o notebook está em `notebooks/`, então o CSV fica em `../data/raw/`).

In [ ]:
# 0.1 Importação das bibliotecas (pandas, numpy, matplotlib, seaborn e os módulos do scikit-learn)

In [ ]:
# 0.2 Leitura do CSV com caminho relativo e visualização das primeiras linhas

---
## Fase 1 — Análise Exploratória (EDA)

**O que esta fase pede:**
- Dimensões do dataset (linhas e colunas)
- Tipos de dados de cada variável
- Resumo estatístico das colunas numéricas com `.describe()`
- **No mínimo 3 gráficos** (Matplotlib/Seaborn), por exemplo:
  1. Histograma das variáveis preditoras
  2. Gráfico de barras com o desbalanceamento da variável alvo
  3. Mapa de calor com a correlação de Pearson
- Célula de texto analisando os números e os gráficos

In [ ]:
# 1.1 Dimensões do dataset (linhas, colunas)

In [ ]:
# 1.2 Tipos de dados das variáveis

In [ ]:
# 1.3 Resumo estatístico descritivo das colunas numéricas (.describe())

In [ ]:
# 1.4 Gráfico 1 — Histogramas das variáveis preditoras numéricas

In [ ]:
# 1.5 Gráfico 2 — Barras com a contagem/proporção de cada classe de falha_maquina

In [ ]:
# 1.6 Gráfico 3 — Mapa de calor da correlação de Pearson entre as variáveis numéricas

### ✍️ Análise

> _Escreva aqui a sua análise. Perguntas para guiar o texto (apague-as depois):_
> - Quantas linhas e colunas a base tem? Algum tipo de dado está errado?
> - Em quais variáveis a média e a mediana (50%) estão distantes? O que isso indica sobre a distribuição?
> - Qual a proporção de falhas (classe 1) e de operação normal (classe 0)? Por que isso exige balanceamento?
> - Quais pares de variáveis têm correlação forte no mapa de calor? Isso afeta a escolha das preditoras?
> - Como essas observações direcionam a estratégia de modelagem (limpeza, balanceamento, escalonamento)?

---
## Fase 2 — Limpeza e Tratamento de Dados (Data Prep)

**O que esta fase pede:**
- Identificar e remover linhas duplicadas
- Identificar dados ausentes e imputar com **média ou mediana**, justificando a escolha pela distribuição
- Boxplots para identificar outliers nas variáveis explicativas

In [ ]:
# 2.1 Quantidade de linhas duplicadas

In [ ]:
# 2.2 Remoção das duplicadas e conferência do novo tamanho da base

#### Justificativa — duplicados

> _Explique o que encontrou e por que a remoção (ou a ausência de duplicados) é importante para o modelo._

In [ ]:
# 2.3 Quantidade de valores ausentes por coluna

In [ ]:
# 2.4 Comparação de média, mediana (e assimetria) das colunas com valores ausentes

#### Justificativa — média ou mediana?

> _Para cada coluna com valores ausentes, diga se usou média ou mediana e por quê, com base no histograma da Fase 1 e na comparação acima. Lembre a regra vista em aula: quando média e mediana ficam muito distantes, a distribuição é assimétrica e a mediana representa melhor o valor típico._

In [ ]:
# 2.5 Imputação dos valores ausentes

In [ ]:
# 2.6 Conferência: não deve restar nenhum valor ausente

In [ ]:
# 2.7 Boxplots das variáveis explicativas para identificar outliers

### ✍️ Análise

> _Escreva aqui a sua análise. Perguntas para guiar o texto (apague-as depois):_
> - Em quais variáveis aparecem pontos fora dos limites do boxplot (1,5 × IQR)?
> - Esses outliers parecem erro de medição ou situações reais de operação, que podem ser justamente as falhas?
> - Você decidiu manter ou tratar os outliers? Justifique.

---
## Fase 3 — Feature Engineering

**O que esta fase pede:**
- Criar uma **nova coluna numérica** por operação matemática entre colunas existentes, depois do tratamento dos nulos
- Sugestão do enunciado: `potencia = velocidade_rotacao_rpm * torque_nm`
- Se escolher outra combinação, explique aqui e no vídeo

In [ ]:
# 3.1 Criação da nova coluna

In [ ]:
# 3.2 Validação: conferir que a nova coluna não tem valores ausentes (NaN) e ver o resumo estatístico

#### Justificativa — nova variável

> _Explique o significado físico da nova coluna e por que ela pode ajudar a prever falhas._

---
## Fase 4 — Divisão e Balanceamento dos Dados

**O que esta fase pede:**
- Separar as preditoras (**X**) da variável alvo (**y**)
- Dividir em **treino (80%)** e **teste (20%)** com `stratify=y`
- Aplicar **Random Under Sampling** **somente no treino**, para evitar vazamento de dados

**Antes de montar o X, decida e justifique:**
- Quais colunas saem: identificadores (`udi`, `id_produto`), a variável alvo e as 5 colunas de motivo da falha (regra da Engenharia)
- O que fazer com a coluna categórica `tipo` (L/M/H), já que os modelos só aceitam números

#### Justificativa — escolha das preditoras

> _Liste as colunas que entraram em X e explique por que cada coluna removida ficou de fora._

In [ ]:
# 4.1 Separação entre X (preditoras) e y (alvo)

In [ ]:
# 4.2 Divisão treino/teste 80/20 com stratify=y (fixe random_state para reprodutibilidade)

In [ ]:
# 4.3 Conferência da proporção das classes no treino e no teste

In [ ]:
# 4.4 Random Under Sampling SOMENTE no treino (classe majoritária reduzida ao tamanho da minoritária)

In [ ]:
# 4.5 Conferência da proporção das classes no treino balanceado

#### Justificativa — divisão e balanceamento

> _Explique: (1) por que usar `stratify=y`; (2) por que balancear só o treino e nunca o teste (data leakage); (3) por que escolheu o Random Under Sampling e qual a desvantagem dele (descartar dados da classe majoritária)._

---
## Fase 5 — Escalonamento de Variáveis (StandardScaler)

**O que esta fase pede:**
- Aplicar o `StandardScaler` **apenas** nas variáveis contínuas usadas pelo **KNN**
- `fit_transform` no **treino** e `transform` no **teste**
- Deixar os dados da **Árvore de Decisão sem escalonamento**, justificando

In [ ]:
# 5.1 StandardScaler: fit_transform no treino balanceado e transform no teste (dados do KNN)

In [ ]:
# 5.2 Conferência: no treino escalonado, as colunas devem ter média ≈ 0 e desvio padrão ≈ 1

#### Justificativa — por que escalonar só para o KNN?

> _Explique: (1) por que o KNN depende da escala (distância entre pontos; compare a ordem de grandeza da rotação em RPM com a do torque em N·m); (2) por que o `fit` é feito só no treino; (3) por que a Árvore de Decisão não é afetada pela escala (ela divide por limiares em uma variável de cada vez)._

---
## Fase 6 — Ajuste de Parâmetros e Combate ao Overfitting

**O que esta fase pede:**
- **KNN:** variar `n_neighbors` (K) em pelo menos **3 valores ímpares** (ex.: 3, 5, 7) e registrar acurácia de **treino** e **teste**
- **Árvore:** variar `max_depth` em pelo menos **3 limites** (ex.: 3, 5, None) e registrar acurácia de **treino** e **teste**
- Texto apontando onde houve overfitting e qual configuração ficou estável no teste

In [ ]:
# 6.1 KNN — treinar com cada valor de K e registrar acurácia de treino e teste

In [ ]:
# 6.2 KNN — tabela (DataFrame) com os resultados

In [ ]:
# 6.3 Árvore de Decisão — treinar com cada max_depth e registrar acurácia de treino e teste

In [ ]:
# 6.4 Árvore de Decisão — tabela (DataFrame) com os resultados

In [ ]:
# 6.5 (Opcional) Gráfico comparando acurácia de treino x teste para cada configuração

### ✍️ Análise

> _Escreva aqui a sua análise. Perguntas para guiar o texto (apague-as depois):_
> - Em quais configurações a acurácia de treino ficou muito acima da de teste (sinal de overfitting)?
> - Como o K pequeno no KNN e o max_depth=None na Árvore se comportaram? Por quê?
> - Qual configuração de cada modelo teve o melhor equilíbrio entre treino e teste?

---
## Fase 7 — Avaliação da Acurácia e Veredito Final

**O que esta fase pede:**
- Calcular e exibir a **acurácia final no teste** do **melhor KNN** e da **melhor Árvore de Decisão**
- Comparar as taxas de acerto e concluir qual modelo a empresa deve adotar

In [ ]:
# 7.1 Treinar o melhor KNN e calcular a acurácia no teste

In [ ]:
# 7.2 Treinar a melhor Árvore de Decisão e calcular a acurácia no teste

In [ ]:
# 7.3 Comparação lado a lado das acurácias

### ✍️ Veredito final

> _Compare os valores numéricos de acurácia no teste, indique o modelo de maior desempenho e justifique por que ele deve ser adotado pela empresa. Se quiser, comente as limitações (por exemplo, a acurácia sozinha em um problema de falhas raras)._